In [1]:
from peft import PeftModel
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer
from rouge_score import rouge_scorer
import pandas as pd
import torch

c:\Users\User\anaconda3\envs\summarization\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
validation = pd.read_pickle('validation_clean.pkl')

In [3]:
model_name = 'google/flan-t5-large'
model_base = AutoModelForSeq2SeqLM.from_pretrained(model_name)
model = PeftModel.from_pretrained(model_base, "./flan-t5-finetuned/checkpoint-70")
tokenizer = AutoTokenizer.from_pretrained("./flan-t5-finetuned/checkpoint-70")

model.eval()
model.to("cuda")

Loading weights: 100%|██████████| 558/558 [00:00<00:00, 2780.37it/s]
[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


PeftModelForSeq2SeqLM(
  (base_model): LoraModel(
    (model): T5ForConditionalGeneration(
      (shared): Embedding(32128, 1024)
      (encoder): T5Stack(
        (embed_tokens): Embedding(32128, 1024)
        (block): ModuleList(
          (0): T5Block(
            (layer): ModuleList(
              (0): T5LayerSelfAttention(
                (SelfAttention): T5Attention(
                  (q): lora.Linear(
                    (base_layer): Linear(in_features=1024, out_features=1024, bias=False)
                    (lora_dropout): ModuleDict(
                      (default): Dropout(p=1e-05, inplace=False)
                    )
                    (lora_A): ModuleDict(
                      (default): Linear(in_features=1024, out_features=16, bias=False)
                    )
                    (lora_B): ModuleDict(
                      (default): Linear(in_features=16, out_features=1024, bias=False)
                    )
                    (lora_embedding_A): ParameterDict()
     

In [4]:
scorer = rouge_scorer.RougeScorer(["rouge1", "rouge2", "rougeL"], use_stemmer=True)

In [ ]:
preds = []
summaries = []

for texto, sum in zip(validation['text'][:50], validation['summary']):
    inputs = tokenizer(texto, max_length = 1024, return_tensors="pt", truncation=False, padding='max_length').to('cuda')

    with torch.no_grad():
        output = model.generate(
        **inputs,
        max_new_tokens=1024,
        min_new_tokens=512,
        num_beams=4,
        length_penalty=1.2,
        early_stopping=True
)

    pred = tokenizer.decode(output[0], skip_special_tokens=True)
    preds.append(pred)
    summaries.append(sum)


for pred,sum in zip(preds[:5], summaries):
    scores = scorer.score(pred,sum)
    print(f"ROUGE-L: {scores['rougeL'].fmeasure:4f} | ROUGE-1: {scores['rouge1'].fmeasure:4f} | ROUGE-2: {scores['rouge2'].fmeasure:4f} ")

In [ ]:
texto = validation['text'][3]

inputs = tokenizer(texto, max_length = 1024, return_tensors="pt", truncation=False, padding='max_length').to('cuda')

with torch.no_grad():
        output = model.generate(
        **inputs,
        max_new_tokens=1024,
        min_new_tokens=512,
        num_beams=4,
        length_penalty=1.2,
        early_stopping=True
    )
pred = tokenizer.decode(output[0], skip_special_tokens=True)


In [ ]:
print(pred)

The SRHR Support Index has potential to broaden SRHR attitude research from a comprehensive perspective addressing the need for a common measure to track progress over time in contexts where such data are limited, including sub-Saharan Africa. While the index performed well across countries and socioeconomic subgroups and were combined into a comprehensive SRHR Support Index , standardized on a 1-100 scale mean 39.19, SD 15.27, Cronbach s alpha 0.80 with higher values The SRHR Support Index has potential to broaden SRHR attitude research from a comprehensive perspective addressing the need for a common measure to track progress over time in contexts where such data are limited, including sub-Saharan Africa. While the index performed well, further validation is needed to assess its applicability in different settings and populations. Introduction Sexual and reproductive as married women or adolescents 9-12 . Consequently, there is a need for new, comprehensive measures using nationally 

In [ ]:
print(validation['summary'][3])

Introduction
Addressing attitudes is central to achieving sexual and reproductive health and rights (SRHR) as per the 2030 Agenda. We aimed to develop a comprehensive index to measure individual’s support for SRHR, expanding opportunities for global trend analyses and tailored interventions. 
Methods
We used nationally representative data on attitudes towards different dimensions of SRHR, collected via a new module integrated into the World Values Survey in Ethiopia, Kenya, and Zimbabwe during 2020–2021 (N=3,711). Exploratory factor analysis of 58 items was used to identify sub-scales, combined into an overall index. Adjusted regression models were used to evaluate the index according to sociodemographic characteristics, stratified by country and sex.
Results
A 23-item, five-factor solution was identified and used to construct sub-indices reflecting support for: (1) sexual and reproductive rights, (2) neighborhood sexual safety, (3) gender-equitable relationships, (4) masculinity norms